In [107]:

# ============================================================
# PHASE 1 - EXPLORATION DU REFERENTIEL TVA
# Objectif : observer les données sans les corriger
# ============================================================

import pandas as pd
from IPython.display import display, Markdown

# ------------------------------------------------------------
# 0. CHARGEMENT ET DECOUVERTE DES DONNEES
# ------------------------------------------------------------

display(Markdown("# Phase 1 — Exploration du référentiel TVA"))
display(Markdown(
    "Objectif : analyser les données brutes avant de définir "
    "les règles de nettoyage. Aucune correction n'est effectuée."
))

chemin_csv = "../starter-kit/data/numeros_tva.csv"
chemin_excel = "../starter-kit/data/numeros_tva.xlsx"

df = pd.read_csv(
    chemin_csv,
    dtype=str,
    keep_default_na=False
)

print("Dimensions du référentiel :", df.shape)
print("\nColonnes :", df.columns.tolist())

display(Markdown("### Premières lignes"))
display(df.head())

display(Markdown("### Dernières lignes"))
display(df.tail())

print("\nInformations générales :")
df.info()


# ------------------------------------------------------------
# QUESTION 1 - PAYS DISTINCTS ET FORMATS
# ------------------------------------------------------------

display(Markdown("## 1. Pays distincts et formats des numéros"))

nombre_pays = df["pays_declare"].nunique()

print("Nombre de pays distincts :", nombre_pays)
print("\nRépartition par pays :")
display(df["pays_declare"].value_counts().to_frame("nombre"))

# Transformer chaque numéro en modèle de format :
# L = lettre, C = chiffre, autres caractères conservés

def identifier_format(numero):
    resultat = ""

    for caractere in numero:
        if caractere.isdigit():
            resultat += "C"
        elif caractere.isalpha():
            resultat += "L"
        else:
            resultat += caractere

    return resultat

formats = df["numero_tva"].apply(identifier_format)

print("Nombre total de formats distincts :", formats.nunique())

# Nombre de formats par pays
formats_par_pays = (
    pd.DataFrame({
        "pays": df["pays_declare"],
        "format": formats
    })
    .groupby("pays")["format"]
    .nunique()
    .sort_values(ascending=False)
)

display(Markdown("### Nombre de formats par pays"))
display(formats_par_pays.to_frame("formats_distincts"))

# Zoom sur les numéros français
fr = df[df["pays_declare"] == "FR"]

print("\nNombre de lignes FR :", len(fr))
print("Nombre de formats FR :",
      fr["numero_tva"].apply(identifier_format).nunique())

print("Numéros FR avec tiret :",
      fr["numero_tva"].str.contains("-", regex=False).sum())

print("Numéros FR avec point :",
      fr["numero_tva"].str.contains(".", regex=False).sum())

print("Numéros FR avec espace :",
      fr["numero_tva"].str.contains(" ", regex=False).sum())

print("Numéros commençant par fr :",
      fr["numero_tva"].str.startswith("fr").sum())

print("Numéros commençant par FR :",
      fr["numero_tva"].str.startswith("FR").sum())

display(Markdown("""
**Observations initiales :**

Le référentiel contient 15 codes pays distincts, alors que le
module de validation couvre 10 pays.

Certains codes, comme `ZZ`, `XX` et `QQ`, semblent inhabituels.
Les codes `GB` et `UK` nécessitent également une vérification.

Pour les 974 lignes déclarées françaises, plusieurs présentations
ont été observées : 54 avec tiret, 46 avec point, 109 avec espace,
44 commençant par `fr` et 799 par `FR`.

Les résultats affichés ci-dessus permettent également de connaître
le nombre exact de formats distincts.
"""))


# ------------------------------------------------------------
# QUESTION 2 - CARACTERES INHABITUELS
# ------------------------------------------------------------

display(Markdown("## 2. Caractères inhabituels"))

masque_caracteres = df["numero_tva"].str.contains(
    r"[^A-Za-z0-9]",
    regex=True
)

nombre_caracteres = int(masque_caracteres.sum())
pourcentage_caracteres = masque_caracteres.mean() * 100

print("Lignes concernées :", nombre_caracteres)
print("Pourcentage :", round(pourcentage_caracteres, 2), "%")

display(Markdown("### Exemples de numéros concernés"))
display(
    df.loc[masque_caracteres, ["numero_tva", "pays_declare"]]
    .head(10)
)

display(Markdown(f"""
**Observation :**

{nombre_caracteres} lignes sur {len(df)} contiennent au moins
un caractère non alphanumérique, soit
**{pourcentage_caracteres:.2f} %** du référentiel.

Ces caractères peuvent être des espaces, des tirets, des points
ou d'autres caractères particuliers.
"""))


# ------------------------------------------------------------
# QUESTION 3 - VALEURS VIDES
# ------------------------------------------------------------

display(Markdown("## 3. Valeurs vides"))

# Conserver les écritures originales pour les observer
valeurs = df["numero_tva"]

vides = [
    "", "NULL", "NONE", "N/A", "NA",
    "-", ".", "VIDE", "NAN"
]

masque_vides = (
    valeurs.str.strip().str.upper().isin(vides)
)

nombre_vides = int(masque_vides.sum())
pourcentage_vides = masque_vides.mean() * 100

print("Nombre de valeurs assimilables à du vide :", nombre_vides)
print("Pourcentage :", round(pourcentage_vides, 2), "%")

print("\nFormes exactes observées :")
display(
    valeurs[masque_vides]
    .apply(repr)
    .value_counts()
    .to_frame("nombre")
)

display(Markdown("""
**Observations initiales :**

Le référentiel contient 260 valeurs assimilables à du vide,
soit 2,6 % des lignes, réparties sous cinq formes observées :

- 60 chaînes vides ;
- 59 valeurs composées d'espaces ;
- 55 tirets ;
- 48 valeurs `N/A` ;
- 38 valeurs `null`.

L'option `keep_default_na=False` permet de conserver
ces représentations lors du chargement du CSV.
"""))


# ------------------------------------------------------------
# QUESTION 4 - PAYS DECLARE ET PREFIXE TVA
# ------------------------------------------------------------

display(Markdown("## 4. Correspondance pays / préfixe TVA"))

# Comparer les deux premiers caractères sans les modifier
prefixes = df["numero_tva"].str.slice(0, 2)

masque_differences = df["pays_declare"] != prefixes

nombre_differences = int(masque_differences.sum())
pourcentage_differences = masque_differences.mean() * 100

print("Nombre de différences :", nombre_differences)
print("Pourcentage :", round(pourcentage_differences, 2), "%")

display(Markdown("### Exemples de différences"))
display(
    df.loc[
        masque_differences,
        ["pays_declare", "numero_tva"]
    ].head(10)
)

display(Markdown(f"""
**Observation :**

Sur {len(df)} lignes, **{nombre_differences}**
({pourcentage_differences:.2f} %) présentent une différence
entre le pays déclaré et les deux premiers caractères
du numéro de TVA.

Ces écarts incluent notamment les préfixes absents,
les valeurs manquantes et les différences de casse.

Le pays déclaré ne correspond donc pas systématiquement
au préfixe du numéro.
"""))


# ------------------------------------------------------------
# QUESTION 5 - COMPARAISON CSV ET EXCEL
# ------------------------------------------------------------

display(Markdown("## 5. Comparaison CSV et Excel"))

df_excel = pd.read_excel(
    chemin_excel,
    dtype=str,
    keep_default_na=False
)

print("Dimensions CSV :", df.shape)
print("Dimensions Excel :", df_excel.shape)

memes_colonnes = list(df.columns) == list(df_excel.columns)

print("Colonnes identiques et dans le même ordre :", memes_colonnes)

if memes_colonnes and df.shape == df_excel.shape:

    comparaison = df.reset_index(drop=True).compare(
        df_excel.reset_index(drop=True)
    )

    print("Nombre de cellules différentes :",
          int((df != df_excel).sum().sum()))

    print("Nombre de lignes différentes :",
          int((df != df_excel).any(axis=1).sum()))

    print("\nDifférences par colonne :")
    display((df != df_excel).sum().to_frame("differences"))

    if comparaison.empty:
        print("Les deux fichiers contiennent les mêmes valeurs.")
    else:
        display(Markdown("### Exemples de différences"))
        display(comparaison.head(10))

else:
    print(
        "Les structures diffèrent : une comparaison directe "
        "ligne par ligne n'est pas possible."
    )

display(Markdown("""
**Observation à finaliser :**

Cette comparaison permet de vérifier si les deux fichiers
contiennent les mêmes données une fois chargés avec pandas.

Le CSV est utilisé comme fichier de travail pour l'exploration.

Le choix définitif de la source de vérité devra être justifié
à partir des différences réellement observées.
"""))


# ------------------------------------------------------------
# QUESTION 6 - CANAUX ET PERIODES DE SAISIE
# ------------------------------------------------------------

display(Markdown("## 6. Canaux et périodes de saisie"))

print("Répartition par canal de saisie :")
display(
    df["source_saisie"]
    .value_counts()
    .to_frame("nombre")
)

print("\nRépartition mensuelle :")

repartition_mensuelle = (
    df["date_saisie"]
    .str.slice(0, 7)
    .value_counts()
    .sort_index()
)

display(repartition_mensuelle.to_frame("nombre"))

# Vérifier aussi les anomalies selon le canal
print("\nCaractères inhabituels par canal :")

analyse_canaux = (
    pd.DataFrame({
        "canal": df["source_saisie"],
        "caracteres_inhabituels": masque_caracteres,
        "valeur_vide": masque_vides
    })
    .groupby("canal")
    .agg(
        total=("caracteres_inhabituels", "size"),
        caracteres_inhabituels=("caracteres_inhabituels", "sum"),
        valeurs_vides=("valeur_vide", "sum")
    )
)

analyse_canaux["taux_caracteres_%"] = (
    analyse_canaux["caracteres_inhabituels"]
    / analyse_canaux["total"] * 100
).round(2)

analyse_canaux["taux_vides_%"] = (
    analyse_canaux["valeurs_vides"]
    / analyse_canaux["total"] * 100
).round(2)

display(analyse_canaux)

display(Markdown("""
**Observations initiales :**

Les cinq canaux de saisie présentent des volumes
relativement équilibrés, entre 1 944 et 2 071 enregistrements.

Les saisies mensuelles sont globalement régulières entre
janvier 2024 et septembre 2025.

Octobre 2025 se distingue avec seulement 16 saisies,
ce qui pourrait correspondre à un mois incomplet.

L'analyse des taux d'anomalies par canal permet de vérifier
si certains canaux présentent davantage de problèmes
malgré des volumes similaires.
"""))


# ------------------------------------------------------------
# CONCLUSION GENERALE
# ------------------------------------------------------------

display(Markdown("""
## Conclusion de la phase 1

L'exploration du référentiel met en évidence plusieurs
points nécessitant une attention particulière :

- des formats de numéros de TVA hétérogènes ;
- des caractères inhabituels ;
- plusieurs représentations des valeurs vides ;
- des incohérences entre le pays déclaré et le préfixe TVA ;
- des différences éventuelles entre les sources CSV et Excel ;
- des variations selon les canaux et les périodes de saisie.

Ces observations permettent de préparer la phase 2,
consacrée au nettoyage et à la validation structurelle.

**Aucune correction des données sources n'a été effectuée
pendant cette phase d'exploration.**
"""))

print("\nPHASE 1 - EXPLORATION TERMINEE")


# Phase 1 — Exploration du référentiel TVA

Objectif : analyser les données brutes avant de définir les règles de nettoyage. Aucune correction n'est effectuée.

Dimensions du référentiel : (10000, 6)

Colonnes : ['id', 'raison_sociale', 'pays_declare', 'numero_tva', 'date_saisie', 'source_saisie']


### Premières lignes

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm


### Dernières lignes

... (35 outputs left out)

,total,caracteres_inhabituels,valeurs_vides,taux_caracteres_%,taux_vides_%
canal,,,,,
crm,2019,385,57,19.07,2.82
import_fournisseur,1944,380,56,19.55,2.88
portail_client,1990,358,47,17.99,2.36
reprise_erp,2071,389,59,18.78,2.85
saisie_manuelle,1976,349,41,17.66,2.07



**Observations initiales :**

Les cinq canaux de saisie présentent des volumes
relativement équilibrés, entre 1 944 et 2 071 enregistrements.

Les saisies mensuelles sont globalement régulières entre
janvier 2024 et septembre 2025.

Octobre 2025 se distingue avec seulement 16 saisies,
ce qui pourrait correspondre à un mois incomplet.

L'analyse des taux d'anomalies par canal permet de vérifier
si certains canaux présentent davantage de problèmes
malgré des volumes similaires.



## Conclusion de la phase 1

L'exploration du référentiel met en évidence plusieurs
points nécessitant une attention particulière :

- des formats de numéros de TVA hétérogènes ;
- des caractères inhabituels ;
- plusieurs représentations des valeurs vides ;
- des incohérences entre le pays déclaré et le préfixe TVA ;
- des différences éventuelles entre les sources CSV et Excel ;
- des variations selon les canaux et les périodes de saisie.

Ces observations permettent de préparer la phase 2,
consacrée au nettoyage et à la validation structurelle.

**Aucune correction des données sources n'a été effectuée
pendant cette phase d'exploration.**



PHASE 1 - EXPLORATION TERMINEE
